# Train the Aircraft Vision model (free Colab)

Everything is saved to your Google Drive, so **if Colab stops (usage limit, disconnect, closed tab), nothing is lost.** Just come back later and do **Runtime → Run all** again. It skips what's done and carries on training from the last finished round.

**Step 1: download the photos (no GPU needed, so it doesn't use up your GPU time)**
- Runtime → Change runtime type → **CPU** → Save, then **Runtime → Run all**.
- Allow it to connect to Google Drive when asked.
- It stops at the "Train" cell with a message telling you to switch to GPU. That's expected.

**Step 2: train (needs GPU)**
- Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**.
- If it stops because of the usage limit, wait until GPU is available again (often the next day) and **Run all** again.

At the end it downloads `aircraft_model.zip`.

In [ ]:
import os
os.environ['PYTHONUNBUFFERED'] = '1'   # show script output live
if not os.path.exists('/content/aircraft-vision'):
    !git clone -q https://github.com/Themagicianleo/aircraft-vision.git /content/aircraft-vision
else:
    !git -C /content/aircraft-vision pull -q
%cd /content/aircraft-vision/training
!pip -q install timm onnx onnxruntime onnxscript

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/aircraft-vision'   # everything that must survive is kept here
os.makedirs(f'{WORK}/data', exist_ok=True)
print('Saving to Google Drive:', WORK)

## Photos 1/2: FGVC-Aircraft (100 types). Done once, about 10 min

In [ ]:
if os.path.exists(f'{WORK}/data/fgvc_prepared/meta.json'):
    print('Already downloaded.')
else:
    from av.data import prepare_fgvc
    prepare_fgvc('/content/fgvc_raw', workers=2)        # big raw download stays on this machine...
    !cp -r /content/fgvc_raw/fgvc_prepared {WORK}/data/   # ...only the shrunk copy (~1 GB) goes to Drive
    print('Done.')

## Photos 2/2: Wikimedia Commons (modern types + more airliner photos). Done once, can take an hour
If it's interrupted, Run all again and it continues. Types are listed in `training/commons_types.csv`.

In [ ]:
!python fetch_commons.py --out {WORK}/data/commons --per-class 250
!echo "Commons photos so far:" && find {WORK}/data/commons -name '*.jpg' | wc -l

## Train (GPU). Picks up from the last finished round if it was interrupted

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('Photos are ready. Now switch to a GPU: Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all.')
!rsync -a {WORK}/data/ /content/data_local/    # local copy: much faster to read than Drive
!python train.py --data-root /content/data_local --extra-data /content/data_local/commons --out {WORK}/runs/effb0 --resume --workers 2 --epochs 40 --img-size 320

## Export to ONNX (fp32 + int8) and measure accuracy with the app's exact preprocessing

In [ ]:
!python export.py --checkpoint {WORK}/runs/effb0/last.pt --out {WORK}/exported --int8 --calib-dir /content/data_local/fgvc_prepared/images
!python evaluate.py --onnx {WORK}/exported/aircraft.onnx --data-root /content/data_local --extra-data /content/data_local/commons
!python evaluate.py --onnx {WORK}/exported/aircraft_int8.onnx --labels {WORK}/exported/labels.json --data-root /content/data_local --extra-data /content/data_local/commons

## Download
Pick the int8 model if its accuracy above is within ~1% of fp32 (it's 4× smaller). Otherwise keep fp32.

In [ ]:
USE_INT8 = False
import shutil
pkg = '/content/aircraft_model'
os.makedirs(pkg, exist_ok=True)
shutil.copy(f'{WORK}/exported/' + ('aircraft_int8.onnx' if USE_INT8 else 'aircraft.onnx'), f'{pkg}/aircraft.onnx')
shutil.copy(f'{WORK}/exported/labels.json', f'{pkg}/labels.json')
shutil.copy(f'{WORK}/data/commons/CREDITS.csv', f'{pkg}/PHOTO_CREDITS.csv')  # keep this: CC licences require crediting authors
shutil.make_archive('/content/aircraft_model', 'zip', pkg)
from google.colab import files
files.download('/content/aircraft_model.zip')